In [10]:
import asyncio

from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.messages import TextMessage
from autogen_core import CancellationToken
from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_ext.tools.http import HttpTool
from autogen_core.models import UserMessage
from autogen_ext.models.ollama import OllamaChatCompletionClient

In [11]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_openai import ChatOpenAI
import json


In [12]:
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")
open_router_api_key= os.getenv("OPENROUTER_API_KEY")
 

In [13]:
model=ChatGroq(model="qwen-qwq-32B")
def llm(input):
    model=ChatGroq(model="qwen-qwq-32B")
    output=model.invoke(input)
    return output.content
print(model)

model1=OpenAIChatCompletionClient(model="gpt-4o")
print(model1)

client=<groq.resources.chat.completions.Completions object at 0x1123a7080> async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x114469850> model_name='qwen-qwq-32B' model_kwargs={} groq_api_key=SecretStr('**********')


In [14]:

open_router_model_client = OpenAIChatCompletionClient(
    base_url="https://openrouter.ai/api/v1",
    model="nvidia/llama-3.1-nemotron-70b-instruct",
    api_key=open_router_api_key,
    model_info={
        "family": 'deepseek',
        "vision": True,
        "function_calling": True,
        "json_output": False
    }
)
response = open_router_model_client.create([UserMessage(content="What is the capital of France?", source="user")])

print(response)

<coroutine object BaseOpenAIChatCompletionClient.create at 0x113aa2a40>


/Users/arunkumar/anaconda3/envs/py312/lib/python3.12/site-packages/autogen_ext/models/openai/_openai_client.py:439: UserWarning: Missing required field 'structured_output' in ModelInfo. This field will be required in a future version of AutoGen.
  validate_model_info(self._model_info)
/var/folders/4m/z36lvyp57r37vs_gdh88c2mc0000gn/T/ipykernel_33479/674734784.py:12: RuntimeWarning: coroutine 'BaseOpenAIChatCompletionClient.create' was never awaited
  response = open_router_model_client.create([UserMessage(content="What is the capital of France?", source="user")])


In [15]:
dsa_solver=AssistantAgent(
    name="Complex_dsa_solver",
    model_client=open_router_model_client,
    description="A DSA_SOLVER",
    system_message="You give code only in python to solve complex dsa problems and it should be under 150 words"
)


code_reviwer=AssistantAgent(
    name="Code_rewiever",
    model_client=open_router_model_client,
    description="A Code Reviwer",
    system_message="You will review the code given by Complex_dsa_solver, check whether the code is optimised and within the limit of 150,optimise it, if it satisfed please Terminate the process"
)

code_editor=AssistantAgent(
    name="Code_editor",
    model_client=open_router_model_client,
    description="A Code editor",
    system_message="You will add some comments to the code given by Code_rewiever where it is required and make sure it should be under 30 words"

)




In [16]:
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_agentchat.messages import TextMessage

team=RoundRobinGroupChat(
    participants=[dsa_solver,code_reviwer,code_editor],
    max_turns=3

)

In [17]:
# async def run_team():
#     task=TextMessage(content="write a heap sort program",source="user")

#     result=await team.run(task=task)
    
#     for i in result.messages:
#         print(f"{i.source}:{i.content}")
    

# await run_team()

TaskResult(messages=[TextMessage(id='d143820c-3320-424c-af40-03b5bcd2ac21', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2025, 7, 13, 2, 3, 56, 873989, tzinfo=datetime.timezone.utc), content='write a heap sort program', type='TextMessage'), TextMessage(id='3f55b254-a287-4cd4-b1de-737cd533130f', source='Complex_dsa_solver', models_usage=RequestUsage(prompt_tokens=40, completion_tokens=298), metadata={}, created_at=datetime.datetime(2025, 7, 13, 2, 4, 7, 682146, tzinfo=datetime.timezone.utc), content='Here\'s a concise Heap Sort implementation in Python, under 150 words:\n\n```python\ndef heapify(arr, n, i):\n    """Maintain heap property."""\n    largest = i\n    left = 2 * i + 1\n    right = 2 * i + 2\n\n    if left < n and arr[i] < arr[left]:\n        largest = left\n\n    if right < n and arr[largest] < arr[right]:\n        largest = right\n\n    if largest != i:\n        arr[i], arr[largest] = arr[largest], arr[i]\n        heapify(arr, n, largest)\n\n\ndef heap_sort(arr):\n    """Sort array using Heap Sort."""\n    n = len(arr)\n\n    # Build Max Heap\n    for i in range(n // 2 - 1, -1, -1):\n        heapify(arr, n, i)\n\n    # Extract elements one by one\n    for i in range(n - 1, 0, -1):\n        arr[i], arr[0] = arr[0], arr[i]\n        heapify(arr, i, 0)\n\n\n# Example Usage\narr = [12, 11, 13, 5, 6, 7]\nheap_sort(arr)\nprint("Sorted array:", arr)\n```\n\n**Output:**\n```\nSorted array: [5, 6, 7, 11, 12, 13]\n```', type='TextMessage'), 


TextMessage(id='5538a7dd-6390-43d2-bba7-1cd0684a183f', source='Code_rewiever', models_usage=RequestUsage(prompt_tokens=362, completion_tokens=928), metadata={}, created_at=datetime.datetime(2025, 7, 13, 2, 4, 32, 554908, tzinfo=datetime.timezone.utc), content='**Analysis & Optimisation Report**\n\n### **Initial Code Assessment**\n\n* **Readability:** (Good) Clear function names, comments, and a straightforward structure.\n* **Optimisation:** (Sufficient for most cases) Standard Heap Sort implementation.\n* **Length:** (Within Limit) Under 150 words (excluding comments and example usage).\n\n### **Code**\n\n(Since the code is already within the limit and well-structured, only minor adjustments for slight readability and no significant optimisations are applicable for general use cases. However, I\'ll provide the same code with additional markdown for readability and offer potential optimisations for very specific scenarios.)\n\n#### **Heap Sort Implementation with Minor Readability Adjustments**\n\n```python\n### Heapify Function\ndef heapify(arr, n, i):\n    """Maintain heap property."""\n    largest = i\n    left, right = 2 * i + 1, 2 * i + 2\n\n    # Check left and right child for largest value\n    if left < n and arr[i] < arr[left]:\n        largest = left\n    if right < n and arr[largest] < arr[right]:\n        largest = right\n\n    # Swap and recurse if necessary\n    if largest != i:\n        arr[i], arr[largest] = arr[largest], arr[i]\n        heapify(arr, n, largest)\n\n\n### Heap Sort Main Function\ndef heap_sort(arr):\n    """Sort array using Heap Sort."""\n    n = len(arr)\n\n    # ### Build Max Heap\n    for i in range(n // 2 - 1, -1, -1):\n        heapify(arr, n, i)\n\n    # ### Extract elements one by one\n    for i in range(n - 1, 0, -1):\n        arr[i], arr[0] = arr[0], arr[i]\n        heapify(arr, i, 0)\n\n\n# **Example Usage**\nif __name__ == "__main__":\n    arr = [12, 11, 13, 5, 6, 7]\n    heap_sort(arr)\n    print("Sorted array:", arr)\n```\n\n### **Optimisations (For Specific Scenarios)**\n\nGiven the original code is already efficient for general use, the following are niche optimisations:\n\n1. **Hybrid Sorting for Small Subarrays**:\n   - **For Very Small Arrays** (e.g., < 10 elements), replace Heap Sort with Insertion Sort for the final extraction steps or entirely for the small array, as Insertion Sort has lower overhead.\n\n2. **In-Place Swaps Optimization**:\n   - **No significant change needed**; the current implementation already uses in-place swaps.\n\n3. **Parallel Heapify (For Multi-Core Systems)**:\n   - **Complexity Increase**: Not recommended for simplicity but can be explored for very large datasets by parallelizing the heapify process for non-overlapping subtrees.\n\n#### **Hybrid Example (Insertion Sort for Small Subarrays)**\n\n```python\ndef insertion_sort(arr, start, end):\n    for i in range(start + 1, end + 1):\n        key = arr[i]\n        j = i - 1\n        while j >= start and arr[j] > key:\n            arr[j + 1] = arr[j]\n            j -= 1\n        arr[j + 1] = key\n\ndef heap_sort_optimised(arr):\n    n = len(arr)\n    # ... (Rest remains the same until extraction)\n\n    # **Hybrid Approach for Extraction**\n    for i in range(n - 1, 9, -1):  # Assuming 10 as the threshold\n        arr[i], arr[0] = arr[0], arr[i]\n        heapify(arr, i, 0)\n    # Switch to Insertion Sort for the remaining small subarray\n    insertion_sort(arr, 0, i)\n\n# **Usage**\nif __name__ == "__main__":\n    arr = [12, 11, 13, 5, 6, 7, 1, 2, 3, 4, 8, 9, 10]\n    heap_sort_optimised(arr)\n    print("Sorted array (Hybrid):", arr)\n```\n\n### **Termination Statement**\n\nSince the original code meets the requirements (under 150 words, readable, and sufficiently optimised for general use), and considering the minor adjustments and optional niche optimisations provided:\n\n**PROCESS TERMINATED AS REQUESTED**', type='TextMessage'), 


TextMessage(id='cdcf954c-0478-4aeb-a850-812a7b766093', source='Code_editor', models_usage=RequestUsage(prompt_tokens=1282, completion_tokens=367), metadata={}, created_at=datetime.datetime(2025, 7, 13, 2, 4, 44, 944527, tzinfo=datetime.timezone.utc), content='It seems like there was a mix-up in your request. You initially asked for:\n\n1. **Adding comments to the code under 30 words** (which wasn\'t fully addressed in the provided analysis)\n2. **Writing a Heap Sort program** (which was provided in the initial code and refined in the analysis)\n\nGiven this, I\'ll address the **first unresolved request** (adding comments under 30 words to the **original** code, not the refined analysis code, for clarity) and provide a **final summary** for the Heap Sort program request.\n\n### **Original Code with Additional Comments (Under 30 words)**\n\n```python\ndef heapify(arr, n, i):\n    """Maintain heap property."""\n    # Identify largest among root and children\n    largest = i\n    left, right = 2 * i + 1, 2 * i + 2\n    # ... (rest of the code remains the same for brevity)\n\ndef heap_sort(arr):\n    """Sort array using Heap Sort."""\n    # Build Max Heap & Extract elements\n    # ... (rest of the code remains the same)\n```\n\n### **Final Summary for Heap Sort Program Request**\n\n| **Aspect** | **Summary** |\n| --- | --- |\n| **Request** | Write a Heap Sort program |\n| **Status** | **COMPLETED** in Initial Code |\n| **Code Readability** | Good |\n| **Optimisation** | Sufficient for most cases |\n| **Length** | Under 150 words (initial code) |\n| **Optimisations Provided** | \n   - Hybrid Sorting for Small Subarrays\n   - Note on Parallel Heapify (complex, for large datasets) | \n| **Termination** | **PROCESS TERMINATED AS REQUESTED** |', type='TextMessage')], 



stop_reason='Maximum number of turns 3 reached.')

In [18]:
# Adding Termination Logic 

from autogen_agentchat.conditions import TextMentionTermination,MaxMessageTermination
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_agentchat.ui import Console

#max Message Termincation stops abruptley

my_termination = TextMentionTermination(text='TERMINATE') | MaxMessageTermination(max_messages=1)

teams1=RoundRobinGroupChat(
    participants=[dsa_solver,code_reviwer,code_editor],
    termination_condition=my_termination,
    max_turns=4
)

await Console(teams1.run_stream() )

async def run_teams():
    task=TextMessage(content="Write a program to print stars as diamond. Terminate",source="user")

    result=await teams1.run(task=task)

    for j in result.messages:
        print(f"{j.source}:{j.content}")

await run_teams()

---------- TextMessage (Complex_dsa_solver) ----------
Here is a Python solution for a complex DSA (Data Structures and Algorithms) problem, all under 150 words:

**Problem:** **Minimum Window Substring** (LeetCode #76)
**Challenge:** Given two strings `s` and `t` of lengths `m` and `n` respectively, return the minimum window substring of `s` which will contain all the characters of `t` in complexity O(m + n).

**Python Solution:**
```python
from collections import defaultdict

def minWindow(s: str, t: str) -> str:
    if not s or not t: return ""
    
    t_count = defaultdict(int)
    for c in t: t_count[c] += 1
    
    required = len(t_count)
    formed = 0
    
    window_counts = defaultdict(int)
    ans = float("inf"), None, None
    
    l = 0
    for r in range(len(s)):
        character = s[r]
        window_counts[character] += 1
        if character in t_count and window_counts[character] == t_count[character]:
            formed += 1
        
        while l <= r and forme

In [19]:
state=await team.save_state()
print(state)

{'type': 'TeamState', 'version': '1.0.0', 'agent_states': {'Complex_dsa_solver': {'type': 'ChatAgentContainerState', 'version': '1.0.0', 'agent_state': {'type': 'AssistantAgentState', 'version': '1.0.0', 'llm_context': {'messages': [{'content': 'Here is a Python solution for a complex DSA (Data Structures and Algorithms) problem, all under 150 words:\n\n**Problem:** **Minimum Window Substring** (LeetCode #76)\n**Challenge:** Given two strings `s` and `t` of lengths `m` and `n` respectively, return the minimum window substring of `s` which will contain all the characters of `t` in complexity O(m + n).\n\n**Python Solution:**\n```python\nfrom collections import defaultdict\n\ndef minWindow(s: str, t: str) -> str:\n    if not s or not t: return ""\n    \n    t_count = defaultdict(int)\n    for c in t: t_count[c] += 1\n    \n    required = len(t_count)\n    formed = 0\n    \n    window_counts = defaultdict(int)\n    ans = float("inf"), None, None\n    \n    l = 0\n    for r in range(len(s)

In [20]:
from autogen_agentchat.ui import Console

await Console(teams1.run_stream())

CancelledError: 

In [21]:
from autogen_core import CancellationToken


# Create a cancellation token.
cancellation_token = CancellationToken()

# Use another coroutine to run the team.
run = asyncio.create_task(
    team.run(
        task="sumarise the programs in bullet points",
        #cancellation_token=cancellation_token,
    )
)

# Cancel the run.
cancellation_token.cancel()

try:
    result = await run  # This will raise a CancelledError.
except asyncio.CancelledError:
    print("Task was cancelled.")


In [29]:
from autogen_agentchat.base import TaskResult


team_2 = RoundRobinGroupChat(
    participants=[dsa_solver, code_reviwer, code_editor],
    termination_condition=my_termination,
    max_turns=6
)


async for message in team_2.run_stream(task="write a nice kabir dhoha poetic lines"):
    print(type(message))
    if isinstance(message,TaskResult):
        print("Stop Reason:",message.stop_reason )
    else:
        print(message.source,message)

    

<class 'autogen_agentchat.messages.TextMessage'>
user id='f697761a-bf17-4c6e-89c7-2b8c0494c4e4' source='user' models_usage=None metadata={} created_at=datetime.datetime(2025, 7, 13, 3, 43, 9, 472312, tzinfo=datetime.timezone.utc) content='write a nice kabir dhoha poetic lines' type='TextMessage'
<class 'autogen_agentchat.base._task.TaskResult'>
Stop Reason: Maximum number of messages 1 reached, current message count: 1


In [30]:
from autogen_agentchat.ui import Console

await Console(team_2.run_stream())

---------- TextMessage (Complex_dsa_solver) ----------
Crafting poetic lines in the style of Kabir, a renowned Indian poet and saint known for his Bhakti movement and Doha (couplet) form, is a delightful task. Here's an attempt to write a Doha (and a few more for depth) in a style inspired by Kabir, on a theme that reflects the intersection of spiritual seeking and the contemporary context of our conversation (wisdom in fleeting moments, like code's ephemeral nature):

### **Doha 1: The Ephemeral Code, The Eternal Seeker**

* **(Kabir-esque Doha)**
बिना ध्यान का कोड जल जाता,
अंतर की आग में सब गति राता।
चलती जीवन की पट्टी पर,
ढूँढो सच्चाई, एक पूंछ पर।

* **Translation & Explanation**
> "Unmindful code burns away,
> In the inner fire, all fleeting night.
> On life's moving loom,
> Seek truth, on a single thread."

* **Meaning**: Reflects on the ephemeral nature of code (and worldly pursuits) without mindfulness, urging the seeker to find eternal truth amidst life's transient weave.

### 

TaskResult(messages=[TextMessage(id='2bbc47ef-bdd4-4b4e-9411-f74e8bc8abc3', source='Complex_dsa_solver', models_usage=RequestUsage(prompt_tokens=2298, completion_tokens=764), metadata={}, created_at=datetime.datetime(2025, 7, 13, 3, 45, 16, 449180, tzinfo=datetime.timezone.utc), content='Crafting poetic lines in the style of Kabir, a renowned Indian poet and saint known for his Bhakti movement and Doha (couplet) form, is a delightful task. Here\'s an attempt to write a Doha (and a few more for depth) in a style inspired by Kabir, on a theme that reflects the intersection of spiritual seeking and the contemporary context of our conversation (wisdom in fleeting moments, like code\'s ephemeral nature):\n\n### **Doha 1: The Ephemeral Code, The Eternal Seeker**\n\n* **(Kabir-esque Doha)**\nबिना ध्यान का कोड जल जाता,\nअंतर की आग में सब गति राता।\nचलती जीवन की पट्टी पर,\nढूँढो सच्चाई, एक पूंछ पर।\n\n* **Translation & Explanation**\n> "Unmindful code burns away,\n> In the inner fire, all fleet

In [47]:
from autogen_agentchat.agents import AssistantAgent
add_1_agent_first = AssistantAgent(
    name = 'add_1_agent_first',
    model_client=model1,
    system_message="Add 1 to the number, first number is 0. Give result as output"
)

add_1_agent_second = AssistantAgent(
    name = 'add_1_agent_second',
    model_client=model1,
    system_message="Add 1 to the number you got from previous run. Give result as output."
)
 
add_1_agent_third = AssistantAgent(
    name = 'add_1_agent_third',
    model_client=model1,
    system_message="Add 1 to the number from previous run. Give result as output."
)

my_increment_team = RoundRobinGroupChat(participants=[add_1_agent_first,add_1_agent_second,add_1_agent_third],max_turns=2)



In [45]:
#use chat gpt for the model_info to provide the console else others model dont work
from autogen_agentchat.ui import Console

await Console(my_increment_team.run_stream())

#it resets it
await my_increment_team.reset()

---------- TextMessage (add_1_agent_first) ----------
The number is 0. Adding 1 to it results in 0 + 1 = 1. The output is 1.
---------- TextMessage (add_1_agent_first) ----------
The number is 0. Adding 1 to it results in 0 + 1 = 1. The output is 1.
---------- TextMessage (add_1_agent_first) ----------
The number is 0. Adding 1 to it results in 0 + 1 = 1. The output is 1.
---------- TextMessage (add_1_agent_first) ----------
The number is 0. Adding 1 to it results in 0 + 1 = 1. The output is 1.
---------- TextMessage (add_1_agent_first) ----------
The number is 0. Adding 1 to it results in 0 + 1 = 1. The output is 1.


In [52]:
my_increment_team1 = RoundRobinGroupChat(participants=[add_1_agent_first],max_turns=5)
await my_increment_team.reset()
await Console(my_increment_team1.run_stream())

---------- TextMessage (add_1_agent_first) ----------
1
---------- TextMessage (add_1_agent_first) ----------
2
---------- TextMessage (add_1_agent_first) ----------
1
---------- TextMessage (add_1_agent_first) ----------
1
---------- TextMessage (add_1_agent_first) ----------
2


TaskResult(messages=[TextMessage(id='279829a0-0865-499a-a946-c67559ec595f', source='add_1_agent_first', models_usage=RequestUsage(prompt_tokens=24, completion_tokens=1), metadata={}, created_at=datetime.datetime(2025, 7, 13, 3, 56, 43, 905654, tzinfo=datetime.timezone.utc), content='1', type='TextMessage'), TextMessage(id='f8f21608-613f-407a-b755-26f82411e359', source='add_1_agent_first', models_usage=RequestUsage(prompt_tokens=29, completion_tokens=1), metadata={}, created_at=datetime.datetime(2025, 7, 13, 3, 56, 44, 423179, tzinfo=datetime.timezone.utc), content='2', type='TextMessage'), TextMessage(id='3a206ff0-d9e0-43d0-b961-e2e40bd362ba', source='add_1_agent_first', models_usage=RequestUsage(prompt_tokens=34, completion_tokens=1), metadata={}, created_at=datetime.datetime(2025, 7, 13, 3, 56, 45, 62628, tzinfo=datetime.timezone.utc), content='1', type='TextMessage'), TextMessage(id='d328c591-3056-4153-9b54-e96c79c29fae', source='add_1_agent_first', models_usage=RequestUsage(prompt_